# L13b: Build and Test a Defensive NWS Client

This lab exercises the complete points-to-hourly-forecast workflow using dependency injection and committed fixtures.

> **Learning objectives**
>
> - Replace a live transport with a fixture-backed test double.
> - Test linked requests and required-field validation.
> - Exercise non-success, malformed, and incomplete responses.
> - Keep an optional live integration call outside the required path.


## Setup


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


In [2]:
data_dir = normpath(joinpath(@__DIR__, "..", "data"))
points_body = read(joinpath(data_dir, "nws-points-ithaca.fixture.json"), String)
forecast_body = read(joinpath(data_dir, "nws-forecast-hourly-ithaca.fixture.json"), String)
user_agent = "CHEME-4800-5800-Fall-2026 course@example.edu"


"CHEME-4800-5800-Fall-2026 course@example.edu"

## Inject the transport

`fetch_hourly_forecast` accepts a `getter` function. Production uses the live downloader; this test double records URLs and returns deterministic responses.


In [3]:
calls = String[]
function fixture_get(url::String; user_agent::String)
    push!(calls, url)
    isempty(strip(user_agent)) && throw(ArgumentError("missing User-Agent"))
    return occursin("/points/", url) ?
        HTTPResponse(200, points_body) : HTTPResponse(200, forecast_body)
end

forecast = fetch_hourly_forecast(
    42.443961,
    -76.501881;
    user_agent = user_agent,
    getter = fixture_get,
)
(request_order = calls, forecast = DataFrame(forecast))


(request_order = ["https://api.weather.gov/points/42.443961,-76.501881", "https://api.weather.gov/gridpoints/BGM/52,99/forecast/hourly"], forecast = 2×8 DataFrame
 Row │ windDirection  shortForecast  startTime                  isDaytime  win ⋯
     │ String         String         String                     Bool       Str ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ NW             Mostly Sunny   2026-08-10T09:00:00-04:00       true  5 m ⋯
   2 │ NW             Partly Cloudy  2026-08-10T10:00:00-04:00       true  6 m
                                                               4 columns omitted)

## Force readable failure paths


In [4]:
failing_get(url::String; user_agent::String) = HTTPResponse(503, "service unavailable")
malformed_get(url::String; user_agent::String) = HTTPResponse(200, "{not-json")

function captured_error(getter)
    try
        request_json(
            "https://api.weather.gov/points/0,0";
            user_agent = user_agent,
            getter = getter,
        )
        return "no error"
    catch error
        return sprint(showerror, error)
    end
end

DataFrame(
    case = ["non-success HTTP status", "malformed JSON"],
    message = [captured_error(failing_get), captured_error(malformed_get)],
)


Row,case,message
,String,String
1,non-success HTTP status,"ArgumentError: HTTP GET failed with status 503 for https://api.weather.gov/points/0,0"
2,malformed JSON,"ArgumentError: response from https://api.weather.gov/points/0,0 was not valid JSON: ArgumentError: invalid JSON at byte position 2 (line 1) parsing type string: ExpectedOpeningQuoteChar\n{not-json...\n ^\n"


## Contract checks


In [5]:
@testset "fixture-backed NWS client" begin
    @test length(forecast) == 2
    @test length(calls) == 2
    @test occursin("/points/", first(calls))
    @test occursin("/forecast/hourly", last(calls))
    @test forecast[1]["temperature"] == 72
    @test_throws ArgumentError parse_points_response(Dict("properties" => Dict()))
    @test_throws ArgumentError build_points_url(95.0, -76.5)
end


Test Summary:             | Pass  Total  Time
fixture-backed NWS client |    7      7  0.5s


Test.DefaultTestSet("fixture-backed NWS client", Any[], 7, false, false, true, 1.786458802052305e9, 1.786458802531428e9, false, "In[5]", Random.Xoshiro(0xce777abd6f2a5292, 0x13f8f1324be4e149, 0x3ff186de700a5577, 0x02a4f62150c2db9f, 0x5c9cab95b860a319))

## Optional live integration

Only after the deterministic checks pass, an instructor may call `fetch_hourly_forecast` with the default transport and a real contact address in the `User-Agent`. That call is intentionally absent from automated execution.

**Interpretation prompt:** Which failures are under our program's control, and which can only be reported or retried?
